# Lab 12: Convolutional Neural Networks

Here we will implement a CNN to classify spoken numerals.

Please follow <a href="https://pytorch.org">these instructions</a> to install pytorch.

Run the first two cells to import the relevant libraries

In [ ]:
!pip3 install torch torchvision torchaudio torchcodec
import os
# Prevents an OpenMP conflict between numpy and torch on macOS; harmless elsewhere
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"
import numpy
import torch

# Run these two blocks to load important libraries and set things up
import torch
from torch import nn   # nn = neural-network building blocks (layers, loss functions, Module base class)
import numpy as np

%matplotlib inline
import matplotlib.pyplot as plt


In [ ]:
# Set the random seed.
# Seeding all three (numpy, torch CPU, torch CUDA) + disabling cudnn's nondeterministic
# kernels is how you get reproducible runs across CPU/GPU.
seed = 42

np.random.seed(seed)
torch.manual_seed(seed)
torch.cuda.manual_seed_all(seed)
torch.cuda.manual_seed(seed)
torch.backends.cudnn.benchmark = False
torch.backends.cudnn.deterministic = True


## 1-D CNNs and model interpretation

In this problem you will train a 1-D CNN on audio data, here to classify spoken numerals (e.g. "one", "two"). Then you will use additional techniques to interpret (since it's audio, one can't really say "visualize", but same idea) what the model is doing. Run these cells first to set things up.

**About the data.** `torchaudio.datasets.SPEECHCOMMANDS` is a PyTorch `Dataset` object wrapping Google's Speech Commands corpus. A `Dataset` in PyTorch is any object with `__len__` and `__getitem__` — indexing into it returns one sample. Here each sample is a tuple `(waveform, sample_rate, label, speaker_id, utterance_number)`. The dataset handles the download, caching, and file I/O for us.



In [ ]:
import IPython.display as ipd

import torchaudio
from torchaudio import datasets as audiodatasets

# We will download the data here ... it's big so it might take a bit
audio_save_dir = './'#SPEECHCOMMANDS_data' # set to wherever you want to keep these files
sc_training = audiodatasets.SPEECHCOMMANDS(audio_save_dir, download=True, subset="training")
sc_validation = audiodatasets.SPEECHCOMMANDS(audio_save_dir, download=True, subset="validation")
sc_testing = audiodatasets.SPEECHCOMMANDS(audio_save_dir, download=True, subset="testing")

In [ ]:
# the audio will be downsampled to 8 kHz to make it easier to work with. just run this
new_sample_rate = 8000
transform = torchaudio.transforms.Resample(orig_freq=16000, new_freq=new_sample_rate)

In [ ]:
# the dataset contains a lot of words, but let's focus on the numbers
# this cell can take some time to run, so just keep waiting if it seems like
# it is stuck
sel_labels = ["zero", "one", "two", "three", "four", "five", "six", "seven", "eight", "nine"]

training_inds = np.array([ii for ii,datum in enumerate(sc_training) if datum[2] in sel_labels])
validation_inds = np.array([ii for ii,datum in enumerate(sc_validation) if datum[2] in sel_labels])
testing_inds = np.array([ii for ii,datum in enumerate(sc_testing) if datum[2] in sel_labels])

In [ ]:
# Utility: pad each sample in a batch to the same length.
# Audio clips have different durations, but tensors in a batch must be the same
# shape. torch.nn.utils.rnn.pad_sequence pads the short ones with zeros at the end.
def pad_sequence(batch):
    # Each item comes in as shape (1, time); .t() transposes to (time, 1) because
    # pad_sequence expects the variable-length axis first.
    batch = [item.t() for item in batch]
    batch = torch.nn.utils.rnn.pad_sequence(batch, batch_first=True, padding_value=0.)
    # Permute back to (batch, channels, time) — the shape Conv1d expects.
    return batch.permute(0, 2, 1)


**`DataLoader` and `collate_fn`.** A `DataLoader` wraps a `Dataset` and yields *batches* during training — it handles shuffling, batching, and (optionally) parallel loading with worker processes. The `collate_fn` argument tells it how to combine a list of individual samples into one batch tensor. We need a custom one here because (a) the raw samples come with extra metadata we don't want and (b) they have different lengths and need padding. `SubsetRandomSampler` restricts each loader to the indices we computed above (only samples whose label is a digit).



In [ ]:
# make data loaders
from torch.utils.data.sampler import SubsetRandomSampler

def collate_fn(batch):
    tensors, targets = [], []
    for waveform, _, label, *_ in batch:
        tensors += [waveform]
        targets += [torch.Tensor([sel_labels.index(label)]).squeeze().long()]

    tensors = pad_sequence(tensors)
    targets = torch.stack(targets)

    return tensors, targets


batch_size = 64

train_loader = torch.utils.data.DataLoader(
    sc_training,
    batch_size=batch_size,
    collate_fn=collate_fn,
    sampler=SubsetRandomSampler(training_inds)
)
val_loader = torch.utils.data.DataLoader(
    sc_validation,
    batch_size=batch_size,
    drop_last=False,
    collate_fn=collate_fn,
    sampler=SubsetRandomSampler(validation_inds)
)
test_loader = torch.utils.data.DataLoader(
    sc_testing,
    batch_size=batch_size,
    drop_last=False,
    collate_fn=collate_fn,
    sampler=SubsetRandomSampler(testing_inds)
)

### Plot the waveform and play the audio for one training sample
Set the x-axis values so that they show seconds.

In [ ]:
# plot the waveform for one example (e.g., sc_training[150])

plt.figure(figsize=(5,2))
print(sc_training[150][0])
plt.plot(sc_training[150][0].t().numpy())
plt.xlabel("Time (seconds)")
plt.axis("on")

# play the audio
ipd.Audio(sc_training[150][0], rate=16000)

### Define model

Here is where we will define our CNN model. Assume you are given the basic number of channels, `n_channels`. This model should have the following layers:

* 1-D convolutional layer with `n_channels` kernels of length 80, with a stride of 16
* 1-D batch norm layer
* ReLU
* 1-D max pooling layer with kernel size 4, stride 4
* 1-D convolutional layer with `n_channels` kernels of size 3, with a stride of 1
* 1-D batch norm layer
* ReLU
* 1-D max pooling layer with same parameters as above
* 1-D convolutional layer with `2*n_channels` kernels of size 3, with a stride of 1
* 1-D batch norm layer
* ReLU
* 1-D max pooling with same parameters as above
* 1-D convolutional layer with `2*n_channels` kernels of size 3, with a stride of 1
* 1-D batch norm layer
* ReLU
* 1-D max pooling with same parameters as above
* average pooling across all remaining timepoints (see `AdaptiveAvgPool1d`)
* flattening
* Linear layer with `2*n_channels` inputs and 10 outputs

In [ ]:
# WordRecognizer: a 1-D CNN for classifying spoken digits.
#
# Pattern for defining a model in PyTorch:
#   1. Subclass nn.Module.
#   2. In __init__, create the layers as attributes of `self`. This registers them,
#      so that model.parameters() knows about their weights.
#   3. Call super().__init__() FIRST, before creating any layers. Forgetting this
#      is a common first-time bug.
#   4. Define forward() to specify how inputs flow through the layers.
#   5. At runtime, call model(x) — NOT model.forward(x) directly — so that hooks
#      (e.g., for .eval() mode) fire correctly.
class WordRecognizer(nn.Module):
    def __init__(self, n_input=1, n_output=10, stride=16, n_channel=32):
        super().__init__()
        # nn.Sequential chains layers in order. Equivalent to writing each one out
        # in forward() with h = layer(h) — just more compact.
        self.layers = nn.Sequential(
            # Conv1d args: (in_channels, out_channels, kernel_size, stride)
            # First layer: 1 -> 32 channels, kernel spans 80 samples (= 5 ms at 16 kHz
            # but here 10 ms because we downsample to 8 kHz), stride 16 = heavy
            # downsampling right away to keep later layers cheap.
            nn.Conv1d(n_input, out_channels=n_channel, kernel_size=80, stride=stride),
            # BatchNorm1d normalizes across the batch dimension per channel.
            # Stabilizes training and often lets you use higher learning rates.
            nn.BatchNorm1d(n_channel),
            nn.ReLU(),
            # MaxPool1d(kernel=4, stride=4) downsamples by 4x, taking the max
            # in each non-overlapping window. Adds local translation invariance.
            nn.MaxPool1d(4, stride=4),
            nn.Conv1d(n_channel, out_channels=n_channel, kernel_size=3, stride=1),
            nn.BatchNorm1d(n_channel),
            nn.ReLU(),
            nn.MaxPool1d(4, stride=4),
            # Double the channel count as we downsample — standard CNN pattern:
            # spatial/temporal resolution shrinks, feature diversity grows.
            nn.Conv1d(n_channel, out_channels=2*n_channel, kernel_size=3, stride=1),
            nn.BatchNorm1d(2*n_channel),
            nn.ReLU(),
            nn.MaxPool1d(4, stride=4),
            nn.Conv1d(2*n_channel, out_channels=2*n_channel, kernel_size=3, stride=1),
            nn.BatchNorm1d(2*n_channel),
            nn.ReLU(),
            nn.MaxPool1d(4, stride=4),
            # Collapse whatever time dimension is left to size 1 by averaging.
            # This gives us a fixed-size vector regardless of input length.
            nn.AdaptiveAvgPool1d(1),
            # Flatten (batch, channels, 1) -> (batch, channels) so we can feed
            # it to a Linear layer.
            nn.Flatten(),
            # Final classifier: 64 -> 10 logits (one per digit class).
            nn.Linear(2*n_channel, n_output)
        )

    # forward() defines the computation graph. Called implicitly when you do model(x).
    def forward(self, x):
        return self.layers(x)


model = WordRecognizer(n_input=1, n_output=len(sel_labels), n_channel=32)
print(model)

# Handy utility: count trainable parameters (those with requires_grad=True).
# Useful for sanity-checking model size.
def count_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

n = count_parameters(model)
print("Number of parameters: %s" % n)


### Write the training and test functions
Remember you need to downsample the data to 8000 Hz (using the `transform` function) before you put it into the model!

In [ ]:
# Run this cell first

# Helper: count how many predictions match the target labels in one batch.
# .squeeze() drops any dimensions of size 1; .eq() is elementwise equality;
# .sum() counts the Trues; .item() pulls a Python scalar out of a 0-D tensor.
def number_of_correct(pred, target):
    return pred.squeeze().eq(target).sum().item()

# Classifier outputs one logit per class; argmax over the class dim = predicted label.
def get_likely_index(tensor):
    return tensor.argmax(dim=-1)

# --- Training ingredients ---
# Optimizer: the algorithm that updates weights given gradients. SGD is the
# classic; Adam (used later for input optimization) is an adaptive variant.
# model.parameters() gives the optimizer the list of tensors it's allowed to change.
optimizer = torch.optim.SGD(model.parameters(), lr=.005)  # Try lr=0.05

# Learning-rate scheduler: multiplies lr by gamma every step_size epochs.
# Common pattern — start with a larger lr, drop it later to fine-tune.
scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=20, gamma=0.1)

# CrossEntropyLoss expects raw logits (not softmaxed) and integer class labels.
# It combines log-softmax + negative log likelihood in one numerically stable op.
lossfunction = nn.CrossEntropyLoss()


In [ ]:
# One pass through the training set = one epoch.
#
# The canonical PyTorch training step is always these four calls, in this order:
#   1. optimizer.zero_grad()   -- clear gradients from the previous batch
#   2. outputs = model(inputs) -- forward pass; builds the autograd graph
#   3. loss.backward()         -- backprop; fills .grad on every parameter
#   4. optimizer.step()        -- apply the update to the weights
#
# Skipping zero_grad is a common bug: PyTorch accumulates gradients by default.
def train_one_epoch(model):
    model.train()
    total_loss = 0
    count = 0
    for inputs, labels in train_loader:
        # Zero out the gradients
        optimizer.zero_grad()

        # Downsample the audio data to 8000 Hz
        transformed_inputs = transform(inputs)

        # Run the model on the inputs to get output
        outputs = model.forward(transformed_inputs)

        # Compute the loss
        loss = lossfunction(outputs, labels)

        # Do backprop (compute derivatives for every step)
        loss.backward()

        # Take a step along the gradients and update
        optimizer.step()
        # loss is a 0-D tensor; .item() pulls the Python float so we can
        # accumulate without keeping the computation graph around.
        total_loss += loss.item()
        count += 1
    print('{:>12s} {:>7.5f}'.format('Train loss:', total_loss/count))


# Evaluation loop.
# In a larger project you'd wrap this in `with torch.no_grad():` to save memory,
# and call model.eval() to disable dropout / switch BatchNorm to inference mode.
def test(model):
    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for inputs, labels  in test_loader:
            transformed_inputs = transform(inputs)
            outputs = model.forward(transformed_inputs)
            total += len(outputs)
            correct += number_of_correct(labels, get_likely_index(outputs))
    print('%s accuracy: %0.3f' % ("Test", correct/total))


### Fit the model! Tweak until you get test accuracy of at least 80%

Try changing `optimizer = torch.optim.SGD(model.parameters(), lr=.005)` with higher learning rate `lr`, try adding more epochs, switch optimizer from SGD to AdamW - `torch.optim.AdamW(model.parameters(), lr=1e-3)`.


In [ ]:
# Main training loop. One iteration of the outer for-loop = one epoch.
# scheduler.step() is called once per epoch to advance the learning-rate schedule.
n_epoch = 2

for epoch in range(n_epoch):
    train_one_epoch(model)
    test(model)
    scheduler.step()


### Create contingency matrix to show performance
`contingency` should be a 10x10 matrix where each row is a predicted label and each column a true label. The value of each element in this matrix should be the number of times a test example with the true label given by the column was assigned the predicted label given by the row. For example, if the predicted label for an example was "seven" but the true label was "six", then you should add one to `contingency[7,6]`. Do this for all examples in the test set.

This is a nice way to visualize how well the model is working.

In [ ]:
# Build a confusion (contingency) matrix on the test set.
all_preds = []
all_targets = []

model.eval()
# Collect predictions and true labels over the full test set.
correct = 0
total = 0

with torch.no_grad():
    for inputs, labels in test_loader:
        transformed_inputs = transform(inputs)
    
        # Apply the model to an input and return output
        outputs = model.forward(transformed_inputs)
    
        predicted = get_likely_index(outputs)
        # .extend unpacks the batch into individual tensor elements so the
        # final lists have one entry per test sample, not one per batch.
        all_preds.extend(predicted)
        all_targets.extend(labels)

# create contingency table
contingency = np.zeros((10, 10))
for i in range(len(all_preds)):
    # .detach() returns a view of the tensor with no gradient tracking —
    # needed because we're about to use it as an index, outside the graph.
    contingency[all_preds[i].detach()][all_targets[i].detach()] += 1

plt.matshow(contingency)
plt.colorbar()


### Create a new model and use it to generate optimized inputs
Finally, let's try to generate new sounds that would maximally activate each of the output units in our `WordRecognizer` model. Input optimization is one type of feature visualization (in this case, "audiolization") that can help understand how a model works.

We will do this by using gradient backpropagation to generate sounds that maximally activate each of the `WordRecognizer` outputs. This requires us to define a new model, `InputOptim`, which has one parameter tensor: `optimized_input`, the input that it is trying to optimize. In its `forward` method, this model should apply the pretrained `WordRecognizer` to its `optimized_input` and return the result. We will then use some optimizer (e.g. Adam) to make the `WordRecognizer` output the desired value. We will repeat this for each of the output classes ("zero", "one", "two", etc.). Then we will check to make sure that this works correctly (after optimization, the `WordRecognizer` should be 100% certain that the optimized input belongs to the desired class) and listen to the resulting sounds.

There are two things that you should consider here:
* How should you initialize `optimized_input`? There are many possibilities, and this choice will greatly affect your result.
* How should you do the optimization? Number of epochs, weight decay, and other optimization choices will also have a big effect on your result.

**The key trick: optimizing the input instead of the weights.** Normally in PyTorch, *weights* are the learnable parameters and the *input* is fixed. Here we flip that: we freeze the trained recognizer's weights and treat the input waveform as the thing to optimize. We do this by making `optimized_input` a tensor with `requires_grad=True`, putting it into the optimizer's parameter list (via a custom `parameters()` method), and then running gradient *ascent* on the recognizer's confidence for a target class (equivalently, gradient descent on the cross-entropy loss against that target).

This is called *activation maximization* or *feature visualization*. This is the audio analog of the classic "what image most activates this neuron" plots in vision CNNs.



In [ ]:
# InputOptim: wraps the pretrained recognizer so we can optimize the input to it.
#
# Two things worth noticing:
#   * self.optimized_input is created with requires_grad=True, meaning PyTorch
#     will track gradients through it — that's what makes it optimizable.
#   * We override parameters() to return ONLY optimized_input. This way, when we
#     pass input_model.parameters() to an optimizer, the optimizer will update
#     the input waveform, NOT the recognizer's weights (which we want to keep
#     frozen from their trained values).
class InputOptim(nn.Module):
    def __init__(self, recognizer_model, input_shape=(1,1,8000)):
        super().__init__()
        self.recognizer_model = recognizer_model
        # Random initialization of the waveform we're going to optimize.
        self.optimized_input = torch.randn(size = input_shape, requires_grad=True)

    def forward(self):
        # Pass the current optimized_input through the frozen recognizer.
        return self.recognizer_model(self.optimized_input)

    # Override: optimizer will see only this one tensor, not the recognizer's weights.
    def parameters(self):
        return [self.optimized_input]


In [ ]:
# Train 10 InputOptim models, one per output class.
# Each run: start from random noise, tweak the waveform to maximize the
# recognizer's probability for the target digit.
targets = torch.arange(10).long()
opt_stims = []

n_epochs = 100

for t in targets:
    input_model = InputOptim(model)
    # Adam + a large lr works well here because we're optimizing in 8000-D input
    # space and the gradients are small. Same training-step pattern as before:
    # zero_grad -> forward -> loss -> backward -> step.
    optimizer = torch.optim.Adam(input_model.parameters(), lr = 1e-1)
    lossfxn = nn.CrossEntropyLoss()

    print("target: ", t)
    for epoch in range(n_epochs):
        optimizer.zero_grad()
        outputs = input_model.forward()
        # CrossEntropyLoss expects (batch, classes) logits and (batch,) labels,
        # so wrap the scalar target t in a length-1 batch dim.
        loss = lossfxn(outputs, torch.unsqueeze(t, 0))
        loss.backward()
        optimizer.step()

    # .detach() to pull the tensor out of the computation graph before converting
    # to numpy. Without it you'd get "Can't call numpy on a tensor that requires grad".
    opt_stims.append(input_model.optimized_input.detach().numpy())


In [ ]:
# finally, use this cell to see model predictions for the optimized inputs & listen to the sounds

def predict(tensor):
    # Use the model to predict the label of the waveform
    logits = model(tensor.unsqueeze(0))
    tensor = get_likely_index(logits)
    tensor = sel_labels[tensor.squeeze().item()]
    return logits.squeeze().detach().numpy(), tensor

for ind in range(10):
    utterance = sel_labels[ind]

    probs, pred = predict(torch.Tensor(opt_stims[ind].reshape(1,8000)))
    print(f"Expected: {utterance}. Predicted: {pred}.")
    
    plt.figure()
    plt.bar(range(10), np.exp(probs) / np.exp(probs).sum())
    plt.xticks(range(10), sel_labels)

    ipd.Audio(opt_stims[ind].squeeze(), rate=8000)

In [ ]:
ipd.Audio(opt_stims[0].squeeze(), rate=8000)

### What happens if you initialize `InputOptim` using a real sound, such as one of the training examples? What does this tell us about how this model works?

You get a sound that is a little different from the original in that it has more static and noise, and perhaps even has a lower pitch. This tells us that the model introduces noise in
order to produce the correct output.